# EEG-based Dementia Risk Classification

This notebook demonstrates a complete pipeline for classifying dementia risk from EEG data using a Convolutional Neural Network (CNN) with PyTorch.\n\nThe steps are:\n1.  **Dependency Installation**: Install necessary libraries like `mne`, `torch`, `pandas`, etc.\n2.  **Data Loading**: Load participant information from `participants.tsv` to determine the dementia risk labels based on APOE genotype.\n3.  **EEG Preprocessing**: Load the raw EEG data, apply filters, and segment it into epochs.\n4.  **PyTorch Dataset and DataLoader**: Create a custom `Dataset` and `DataLoader` for efficient training.\n5.  **CNN Model**: Define and train a simple 1D CNN model.\n6.  **Prediction**: Use the trained model to predict dementia risk on a new EEG file.

### 1. Install Dependencies

In [ ]:
!pip install mne torch pandas numpy scikit-learn matplotlib

### 2. Load and Prepare Participant Data

In [ ]:
import pandas as pd

# Load the participants.tsv file
participants_df = pd.read_csv('participants.tsv', sep='\t')

# Filter for participants in the second phase
eeg_participants_df = participants_df[participants_df['second_phase'] == 1].copy()

# Create the target variable 'dementia_risk'
# 1 if 'e4' is in APOE_haplotype, 0 otherwise
eeg_participants_df['dementia_risk'] = eeg_participants_df['APOE_haplotype'].apply(lambda x: 1 if 'e4' in str(x) else 0)

# Display the first few rows of the new dataframe and the distribution of the target variable
print("Filtered participants with dementia risk:")
print(eeg_participants_df[['participant_id', 'APOE_haplotype', 'dementia_risk']].head())
print("\nDementia Risk Distribution:")
print(eeg_participants_df['dementia_risk'].value_counts())

### 3. Load and Preprocess EEG Data

In [ ]:
import os
import mne

# Path to the data
data_path = '.'

# Get the list of participant IDs from the dataframe
participant_ids = eeg_participants_df['participant_id'].tolist()

# Find the resting-state EEG files for these participants
eeg_files = []
for participant_id in participant_ids:
    subject_path = os.path.join(data_path, participant_id, 'eeg')
    if os.path.isdir(subject_path):
        for f in os.listdir(subject_path):
            if f.endswith('_task-rest_eeg.vhdr'):
                eeg_files.append(os.path.join(subject_path, f))

print(f"Found {len(eeg_files)} EEG files.")
print(eeg_files[:5])

In [ ]:
def preprocess_eeg(file_path):
    # Load the raw data
    raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)
    
    # Set montage
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='warn')
    
    # Apply band-pass filter
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    
    # Apply notch filter
    raw.notch_filter(50., fir_design='firwin', verbose=False)
    
    # Set up and fit ICA
    ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter=800)
    ica.fit(raw)
    
    # Find and exclude EOG artifacts
    # This is a bit tricky as we don't have EOG channels. 
    # We can try to find components based on their topography and time course.
    # For this example, I will assume the first two components capture eye movements.
    # In a real scenario, this would need more careful inspection.
    ica.exclude = [0, 1] 
    
    # Apply ICA
    ica.apply(raw)
    
    # Create epochs
    # Using fixed-length epochs of 2 seconds
    epochs = mne.make_fixed_length_epochs(raw, duration=2, overlap=0, preload=True, verbose=False)
    
    return epochs

In [ ]:
# Preprocess all EEG files
all_epochs_data = []
all_labels = []

for file_path in eeg_files:
    participant_id = file_path.split(os.sep)[1]
    label = eeg_participants_df[eeg_participants_df['participant_id'] == participant_id]['dementia_risk'].values[0]
    
    print(f"Processing {file_path} for participant {participant_id} with label {label}")
    
    try:
        epochs = preprocess_eeg(file_path)
        all_epochs_data.append(epochs.get_data())
        all_labels.extend([label] * len(epochs))
    except Exception as e:
        print(f"  Error processing {file_path}: {e}")

# The result `all_epochs_data` will be a list of numpy arrays, and `all_labels` will be a list of integers.
# These will be used to create the PyTorch dataset.
import numpy as np

all_epochs_data = np.concatenate(all_epochs_data, axis=0)
all_labels = np.array(all_labels)

print(f"\nFinished preprocessing. Shape of all epochs: {all_epochs_data.shape}. Total labels: {len(all_labels)}")

### 4. Create PyTorch Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class EEGDataset(Dataset):
    def __init__(self, epochs, labels):
        self.epochs = torch.tensor(epochs, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.epochs[idx], self.labels[idx]

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    all_epochs_data, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

# Create the PyTorch datasets
train_dataset = EEGDataset(X_train, y_train)
val_dataset = EEGDataset(X_val, y_val)

# Create the DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

### 5. Define and Train a CNN Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# Get input shape from the data
n_channels = X_train.shape[1]
n_times = X_train.shape[2]

class EEG_CNN(nn.Module):
    def __init__(self, num_classes=2, n_channels=62, n_times=2000):
        super(EEG_CNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=n_channels, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        
        # Calculate the size of the flattened layer
        self.flat_size = 32 * (n_times // 4)
        
        self.fc1 = nn.Linear(self.flat_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, self.flat_size) # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = EEG_CNN(num_classes=2, n_channels=n_channels, n_times=n_times)
print(model)

In [ ]:
import torch.optim as optim

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10 # For demonstration purposes

for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
    
    # Print loss for the epoch
    print(f'Epoch {epoch + 1}, Loss: {running_loss / len(train_loader)}')

print('Finished Training')

# Validation loop
correct = 0
total = 0
with torch.no_grad():
    for data in val_loader:
        inputs, labels = data
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the validation data: {100 * correct // total} %')

### 6. Prediction on a New EEG File

In [ ]:
def predict_dementia_risk(eeg_file_path, model):
    # Set the model to evaluation mode
    model.eval()
    
    # Preprocess the EEG file
    try:
        epochs = preprocess_eeg(eeg_file_path)
        epochs_data = torch.tensor(epochs.get_data(), dtype=torch.float32)
    except Exception as e:
        print(f"Error processing {eeg_file_path}: {e}")
        return None
        
    # Make a prediction
    with torch.no_grad():
        outputs = model(epochs_data)
        _, predicted = torch.max(outputs.data, 1)
    
    # Since we are processing one file, we can average the predictions 
    # for all epochs from that file to get a single prediction.
    predicted_label = torch.mode(predicted).values.item()
    
    return predicted_label

In [ ]:
# Get an example file to test the prediction function
# Note: In a real-world scenario, you would use a completely new, unseen file.
example_file = eeg_files[0]
print(f"Using example file: {example_file}")

# Make a prediction
predicted_risk = predict_dementia_risk(example_file, model)

if predicted_risk is not None:
    print(f"Predicted Dementia Risk (0=Low, 1=High): {predicted_risk}")
    
    # Get the actual label for comparison
    participant_id = example_file.split(os.sep)[1]
    actual_risk = eeg_participants_df[eeg_participants_df['participant_id'] == participant_id]['dementia_risk'].values[0]
    print(f"Actual Dementia Risk: {actual_risk}")